# Experiment 025 — 30M Story→Math Developmental Shift

Formal T4×2 run. The public comparison is a fixed ~30M Transformer LLM versus a growing CLM upcycled from the retained 30M TextNCA source. Default shift: 50M tokens at 90% synthetic arithmetic / 10% TinyStories.


In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/experiment-025-story-math-growth'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'

if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=ROOT, check=True)

HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
TREE = subprocess.check_output(['git', 'rev-parse', 'HEAD^{tree}'], cwd=ROOT, text=True).strip()
DIRTY = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=ROOT, text=True).strip()
print({'HEAD': HEAD, 'tree': TREE, 'tracked_dirty': bool(DIRTY)})
assert not DIRTY


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_story_math_shift_30m.py',
    'tests/test_clm_progressive_growth.py',
    'tests/test_growth_router.py',
    '-q',
], cwd=ROOT, check=True)


## Formal run

Run the default 50M-token shift. The two workers run concurrently when both T4s are visible. If the 9.25h worker guard is reached, rerun this cell; periodic checkpoints resume automatically.


In [ ]:
subprocess.run([
    sys.executable,
    'scripts/run_experiment_025_story_math_growth.py',
], cwd=ROOT, check=True)


In [ ]:
import json
from IPython.display import display, Image

OUT = ROOT / 'results' / 'experiment-025-story-math-growth'
decision_path = OUT / 'decision.json'
if decision_path.is_file():
    decision = json.loads(decision_path.read_text())
    print(json.dumps(decision, indent=2))
    for name in ['story-math-performance.png', 'growth-timeline.png']:
        path = OUT / name
        if path.is_file():
            display(Image(filename=str(path)))
else:
    print('The run is partial. Re-run the formal cell to resume.')


## Publish after acceptance

Only run this after inspecting `decision.json` and the figures. It uses the existing `GITHUB_TOKEN` Kaggle secret and pushes curated, non-checkpoint artifacts to `kaggle/experiment-025-story-math-growth-results`.


In [ ]:
# subprocess.run([
#     sys.executable,
#     'scripts/publish_experiment_025_story_math_growth.py',
#     '--push',
# ], cwd=ROOT, check=True)
